In [20]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [21]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2025-02-28 13:27:08--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.2.33, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.1’

book-crossings.zip. 100%[===================>]  24.88M   149MB/s    in 0.2s    

2025-02-28 13:27:08 (149 MB/s) - ‘book-crossings.zip.1’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [22]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding="ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'}
)

df_ratings = pd.read_csv(
    ratings_filename,
    encoding="ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'}
)


In [23]:
# add your code here - filtering ratings

# Count ratings per user and per book (by isbn)
user_rating_counts = df_ratings['user'].value_counts()
isbn_rating_counts = df_ratings['isbn'].value_counts()

# Filter out users with fewer than 200 ratings and books with fewer than 100 ratings.
df_ratings_filtered = df_ratings[
    (df_ratings['user'].isin(user_rating_counts[user_rating_counts >= 200].index)) &
    (df_ratings['isbn'].isin(isbn_rating_counts[isbn_rating_counts >= 100].index))
]

print("Filtered ratings shape:", df_ratings_filtered.shape)


Filtered ratings shape: (49781, 3)


In [24]:
# Create a pivot table where rows are ISBNs, columns are users, and values are ratings.
df_table = df_ratings_filtered.pivot_table(index='isbn', columns='user', values='rating').fillna(0)
print("Pivot table shape:", df_table.shape)


Pivot table shape: (731, 888)


In [25]:
# Replace the ISBN index with the book title for easier lookup.
df_books_indexed = df_books.set_index('isbn')

df_table = df_table.join(df_books_indexed['title'])

df_table.set_index('title', inplace=True)
print("Updated pivot table shape:", df_table.shape)


Updated pivot table shape: (731, 888)


In [26]:
# function to return recommended books - this will be tested

knn_model = NearestNeighbors(n_neighbors=6, metric="cosine", algorithm="brute")
knn_model.fit(df_table.values)

def get_recommends(book=""):
    if book not in df_table.index:
        return [book, []]

    book_vector = df_table.loc[book].values.reshape(1, -1)

    distances, indices = knn_model.kneighbors(book_vector, n_neighbors=6)

    recommended_books = []
    for i in range(1, 6):
        rec_title = df_table.index[indices[0][-i]]
        rec_distance = distances[0][-i]
        recommended_books.append([rec_title, rec_distance])

    return [book, recommended_books]


In [27]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", 0.8016211], ['The Weight of Water', 0.77085835], ['The Surgeon', 0.7699411], ['I Know This Much Is True', 0.7677075], ['The Lovely Bones: A Novel', 0.7234864]]]
You passed the challenge! 🎉🎉🎉🎉🎉
